In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(env_path, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError(f"GROQ_API_KEY not found in {env_path} (cwd={Path.cwd()})")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_api_key,
    temperature=0.7,
    max_tokens=800,
    max_retries=5,
    request_timeout=120,
)

In [4]:
from utils import get_serper_api_key

serper_api_key = get_serper_api_key()
if not serper_api_key:
    raise RuntimeError(f"SERPER_API_KEY not found in {env_path} (cwd={Path.cwd()})")

os.environ["SERPER_API_KEY"] = serper_api_key

In [5]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

In [7]:
venue_coordinator = Agent(
    role="Venue Coordinator",
    goal="Identify and book an appropriate venue based on event requirements.",
    backstory=(
        "With a keen sense of space and "
        "understanding of event logistics, "
        "you excel at finding and securing "
        "the perfect venue that fits the event's theme, "
        "size, and budget constraints."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    llm=llm
)

In [8]:
logistics_manager = Agent(
    role="Logistics Manager",
    goal=("Manage all logistics for the event including catering and equipmen"),
    backstory=(
        "Organized and detail-oriented, "
        "you ensure that every logistical aspect of the event "
        "from catering to equipment setup "
        "is flawlessly executed to create a seamless experience."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    llm=llm,
)

In [9]:
marketing_communication_agent = Agent(
    role="Marketing and Communications Agent",
    goal="Effectively market the event and communicate with participants",
    backstory=(
        "Creative and communicative, "
        "you craft compelling messages and "
        "engage with potential attendees "
        "to maximize event exposure and participation."
    ),
    tools=[search_tool, scrape_tool],
    verbose=True,
    llm=llm,
)

In [10]:
from pydantic import BaseModel


class VenueDetails(BaseModel):
    name: str
    address: str
    capacity: str
    booking_status: str



In [11]:
venue_task = Task(
    description="Find a venue in {event_city} that meets criteria for {event_topic}.",
    expected_output="All the details of a specifically chosen"
    "venue you found to accommodate the event.",
    human_input=True,
    output_json=VenueDetails,
    output_file="venue_details.json",
    agent=venue_coordinator,
)

In [12]:
logistics_task = Task(
    description="Coordinate catering and "
    "equipment for an event "
    "with {expected_participants} participants "
    "on {tentative_date}.",
    expected_output="Confirmation of all logistics arrangements "
    "including catering and equipment setup.",
    human_input=True,
    async_execution=True,
    agent=logistics_manager,
)

In [14]:
marketing_task = Task(
    description="Promote the {event_topic} "
    "aiming to engage at least"
    "{expected_participants} potential attendees.",
    expected_output="Report on marketing activities "
    "and attendee engagement formatted as markdown.",
    async_execution=True,
    output_file="marketing_report.md",
    agent=marketing_communication_agent,
)

In [15]:
crew = Crew(
    agents=[venue_coordinator, logistics_manager, marketing_communication_agent],
    tasks=[venue_task, logistics_task, marketing_task],
    verbose=2,
)

In [17]:
inputs = {
    "event_topic": "Tech innovation Conference",
    "event_description": "A gathering of tech innovators "
    "and industry leaders "
    "to explore future technologies.",
    "event_city": "San Francisco",
    "tentative_date": "2024-09-15",
    "expected_participants": 500,
    "budget": 20000,
    "venue_type": "Conference Hall",
}

result = crew.kickoff(inputs=inputs)

 [DEBUG]: == Working Agent: Venue Coordinator
 [INFO]: == Starting Task: Find a venue in San Francisco that meets criteria for Tech innovation Conference.


> Entering new CrewAgentExecutor chain...
Thought: I need to find a suitable venue in San Francisco for a Tech Innovation Conference. First, I will search the internet for venues in San Francisco that are popular for tech conferences, noting their capacity, amenities, and general pricing to ensure they fit typical conference requirements.

Action: Search the internet
Action Input: {"search_query": "best venues in San Francisco for tech innovation conferences capacity amenities"} 


Search results: Title: Modern Tech Innovation Hub & Event Space
Link: https://www.eventsinminutes.com/venues/san-francisco-ca/modern-tech-innovation-hub-event-space
Snippet: Key amenities and inclusions for events are: Complimentary high-speed Wi-Fi throughout the venue; State-of-the-art presentation and sound systems ...
---
Title: Top 20 Conference & S

2026-09-20 15:42:50,857 - 11916 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
2026-09-20 15:43:26,047 - 11916 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)


Thought: The human feedback indicates I should re-evaluate my work. While Convene 100 Stockton is a strong candidate, the prompt asks for "All the details of a specifically chosen venue." To ensure I have the most robust and comprehensive details for a "Tech Innovation Conference," I should verify if there are any specific tech capabilities or pricing details that might be missing, or if I should check another top-tier venue like Moscone Center (which is the standard for major tech events) to see if it offers more specific "tech innovation" amenities. However, Convene 100 Stockton is often preferred for its modern, boutique-yet-large feel which is popular for tech innovation summits that want to avoid the "corporate" feel of Moscone. Let's double-check the specific A/V and tech capabilities of Convene 100 Stockton to ensure the "Tech Innovation" theme is fully supported, as my previous reading was general. I will search for specific tech specs or reviews mentioning its suitability for 

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `qwen/qwen3.8-27b` in organization `org_01m2wtajwgem9r6g7bvbprbjjn` service tier `on_demand` on input tokens per minute (ITPM): Limit 7000, Requested 7600, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

2026-09-20 15:44:21,259 - 11916 - __init__.py-__init__:216 - ERROR: Failed to export span batch code: None, reason: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Read timed out. (read timeout=30.0)
